# H02 — Linear systems: the oscillator a detector is made of

> **References**
>
> (Saulson 2017), (Oppenheim et al. 1997), (Allen and others 2000)

## What you do here

A mirror is suspended from a spring and the ground under it moves. We want the motion of the mirror for any motion of the ground. We kick the mirror once, record how it rings and take the Fourier transform of the ringing: this is the transfer function $G(f)$. With $G(f)$ we find the output for any input, because $X_{\mathrm{out}}(f) = G(f)\,X_{\mathrm{in}}(f)$.

The picture below is the whole workflow. The tasks follow it in order. All the code is given. In every task you change the parameters under `# change these`, run the cells again and answer one question on the physics.

> **The symbols**
>
> - **$m$** — the mass of the mirror, in kg
> - **$k$** — the spring constant, in N/m
> - **$b$** — the damping coefficient, in kg/s: the friction on the mirror is $-b\,\dot{x}_{\mathrm{out}}$
> - **$x_{\mathrm{in}}(t)$** — the input: the motion of the ground, which the suspension point follows
> - **$x_{\mathrm{out}}(t)$** — the output: the motion of the mirror
> - **$g(t)$** — the impulse response: the motion of the mirror after a kick
> - **$G(f)$** — the transfer function: the Fourier transform of $g(t)$; $X_{\mathrm{in}}(f)$ and $X_{\mathrm{out}}(f)$ are the transforms of the input and of the output

| Part | The question | Tasks |
|---|---|---|
| A · The oscillator and its parameters | What do $m$, $k$ and $b$ set? | 1 |
| B · The transfer function from an impulse | How does one kick give $G$? How do we use $G$ for any input? | 2 and 3 |
| C · The resonance and $Q$ | What do we gain and what do we lose with a high $Q$? | 4 |
| D · At the resonance and away from it | When do we work at the resonance and when far above it? | 5 and 6 |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.integrate import solve_ivp
import sys, os

# the course root holds gwcourse_palette.py; find it from wherever we are
here = os.path.abspath('.')
while not os.path.exists(os.path.join(here, 'gwcourse_palette.py')):
    if os.path.dirname(here) == here:
        raise FileNotFoundError('gwcourse_palette.py not found: open the notebook from the course folder')
    here = os.path.dirname(here)
sys.path.insert(0, here)

from gwcourse_palette import P, apply_style
apply_style()

The workflow of this laboratory: the mirror suspended from its spring and the two ways to use it.

![The mirror is suspended from a spring and a damper under a moving support. The same system as a chain, ground, system, mirror. Way 1, kick the mirror, g(t) in time, G(f) in frequency. Way 2, the output for any input, X_out(f) = G(f) X_in(f).](figures/h02_workflow.png)


> **How to use this notebook**
>
> Run the cells in order. Every task has the same parts: what we do, the functions of the task, a cell with the parameters under `# change these`, a box **What to change**, a question and a solution. A function is defined in the task where it is first needed. A later task that uses it says so. Change one parameter at a time and run the cell again. Answer the question before you open the solution. Every cell runs in less than 20 seconds.

## Part A · The oscillator and its parameters

Everything that follows is built on one oscillator. Before we kick it or shake it, we see what its three parameters do.

### Task 1 · What $m$, $k$ and $b$ do

**What we do.** The ground is at rest. We pull the mirror to $x_{\mathrm{out}} = 1$ and let it go. Newton's second law is $m\,\ddot{x}_{\mathrm{out}} + b\,\dot{x}_{\mathrm{out}} + k\,x_{\mathrm{out}} = 0$. The mirror rings and the ringing fades. Three numbers describe it:

$$
\omega_{0} = \sqrt{\frac{k}{m}},
\qquad
Q = \frac{\sqrt{km}}{b},
\qquad
\tau_{0} = \frac{2Q}{\omega_{0}} = \frac{2m}{b} .
$$

The cell draws two plots. In plot (a) the mass and the spring are the same and we use three values of $b$: three values of $Q$. In plot (b) $Q$ is the same and we use three springs. The dashed lines are the envelope $\pm\,\mathrm{e}^{-t/\tau_{0}}$. The cell prints the numbers of each curve.

> **The symbols**
>
> - **$\omega_{0}$** — the natural frequency, in radians per second; $f_{0} = \omega_{0}/2\pi$ is the same in hertz
> - **$Q$** — the quality factor, no unit
> - **$\tau_{0}$** — the ringdown time, in seconds: the time in which the amplitude falls by a factor $\mathrm{e}$

In [ ]:
def oscillator_parameters(m, k, b):
    """Return the natural frequency, the quality factor and the decay time of the oscillator.

    Parameters
    ----------
    m : float, mass of the mirror (kg)
    k : float, spring constant (N/m)
    b : float, damping coefficient (kg/s); b = 0 means no friction

    Returns
    -------
    (f0, q, tau0) : f0 natural frequency (Hz), q quality factor (no unit),
                    tau0 ring-down time (s), the time in which the amplitude falls by e
    """
    # natural frequency in rad/s
    w0 = np.sqrt(k / m)
    # without friction the quality factor and the decay time are infinite
    q = np.sqrt(k * m) / b if b > 0 else np.inf
    tau0 = 2 * m / b if b > 0 else np.inf
    params = (w0 / (2 * np.pi), q, tau0)
    return params


def rhs(t, state, m, k, b, t_grid, x_in_grid):
    """Return the time derivatives of the state: Newton's second law in the form the solver wants.

    Parameters
    ----------
    t         : float, time (s)
    state     : (x_out, v), position and velocity of the mirror (m, m/s)
    m, k, b   : mass (kg), spring constant (N/m), damping coefficient (kg/s)
    t_grid    : array, the times at which the ground position is given (s)
    x_in_grid : array, the ground position x_in at those times (m)

    Returns
    -------
    [v, a] : velocity (m/s) and acceleration (m/s^2) of the mirror
    """
    # the ground position at this time, joining the given samples with straight lines
    x_in = np.interp(t, t_grid, x_in_grid)
    x_out, v = state
    # Newton's second law: the spring pulls on the stretch, the friction on the velocity
    a = (-k * (x_out - x_in) - b * v) / m
    derivatives = [v, a]
    return derivatives


def integrate_oscillator(t, m, k, b, x_in=None, x0=0.0, v0=0.0):
    """Return the motion x_out(t) of the mirror, from Newton's second law.

    Parameters
    ----------
    t       : array, times at which we want the motion, evenly spaced (s)
    m, k, b : mass (kg), spring constant (N/m), damping coefficient (kg/s)
    x_in    : array of the same length as t, the ground position (m); None means a ground at rest
    x0, v0  : position (m) and velocity (m/s) of the mirror at the first time

    Returns
    -------
    x_out : array of the same length as t, the position of the mirror (m)
    """
    # a ground at rest is a ground position of zero at every time
    if x_in is None:
        x_in = np.zeros_like(t)
    # the solver follows (x_out, v) from the start state and gives them at the times t
    sol = solve_ivp(rhs, (t[0], t[-1]), [x0, v0], t_eval=t, args=(m, k, b, t, x_in),
                    method='RK45', rtol=1e-8, atol=1e-10)
    x_out = sol.y[0]
    return x_out

In [ ]:
# change these: (a) the same m and k with three dampings, (b) the same Q with three springs
m = 1.0                          # mass of the mirror (kg)
k = 39.5                         # spring constant of (a) (N/m): f0 = 1 Hz
b_values = (1.26, 0.63, 0.31)    # damping coefficients of (a) (kg/s): Q = 5, 10, 20
k_values = (9.87, 39.5, 158.0)   # spring constants of (b) (N/m): f0 = 0.5, 1, 2 Hz
Q_b = 10.0                       # quality factor of (b), the same for the three springs (no unit)

# (a) let the mirror go from x = 1 with zero velocity, for each damping
params_a = [oscillator_parameters(m, k, b) for b in b_values]
t_a = np.arange(0, 2.5 * max(p[2] for p in params_a), 0.01)
x_a = [integrate_oscillator(t_a, m, k, b, x0=1.0) for b in b_values]

# (b) the same, with the damping that keeps Q fixed: Q = sqrt(k m) / b
b_b = [np.sqrt(k_i * m) / Q_b for k_i in k_values]
params_b = [oscillator_parameters(m, k_i, b_i) for k_i, b_i in zip(k_values, b_b)]
t_b = np.arange(0, 2.5 * max(p[2] for p in params_b), 0.01)
x_b = [integrate_oscillator(t_b, m, k_i, b_i, x0=1.0) for k_i, b_i in zip(k_values, b_b)]

# the numbers of each curve: the swings in one decay time are tau0 * f0
print("(a)  b (kg/s)    f0 (Hz)       Q    tau0 (s)   swings in tau0")
for b_i, (f0_i, Q_i, tau_i) in zip(b_values, params_a):
    print(f"     {b_i:8.2f}  {f0_i:9.3f}  {Q_i:6.1f}  {tau_i:9.2f}  {tau_i * f0_i:10.2f}")
print("(b)  k (N/m)     f0 (Hz)       Q    tau0 (s)   swings in tau0")
for k_i, (f0_i, Q_i, tau_i) in zip(k_values, params_b):
    print(f"     {k_i:8.2f}  {f0_i:9.3f}  {Q_i:6.1f}  {tau_i:9.2f}  {tau_i * f0_i:10.2f}")

In [ ]:
# two plots, each with three curves on the same axes; dashed: the envelope exp(-t / tau0)
colors = [P['BLUE'], P['BRICK'], P['SAGE']]
fig, (axa, axb) = plt.subplots(2, 1, figsize=(9, 7.5))
for x_i, (f0_i, Q_i, tau_i), c in zip(x_a, params_a, colors):
    axa.plot(t_a, x_i, color=c, lw=1.3, label=f'$Q$ = {Q_i:.0f},  $f_0$ = {f0_i:.2f} Hz')
    axa.plot(t_a, np.exp(-t_a / tau_i), color=c, lw=1.0, ls='--')
    axa.plot(t_a, -np.exp(-t_a / tau_i), color=c, lw=1.0, ls='--')
for x_i, (f0_i, Q_i, tau_i), c in zip(x_b, params_b, colors):
    axb.plot(t_b, x_i, color=c, lw=1.3, label=f'$Q$ = {Q_i:.0f},  $f_0$ = {f0_i:.2f} Hz')
    axb.plot(t_b, np.exp(-t_b / tau_i), color=c, lw=1.0, ls='--')
    axb.plot(t_b, -np.exp(-t_b / tau_i), color=c, lw=1.0, ls='--')
axa.set(xlabel='time (s)', ylabel=r'$x_{\mathrm{out}}$ (arbitrary units)', xlim=(0, t_a[-1]))
axb.set(xlabel='time (s)', ylabel=r'$x_{\mathrm{out}}$ (arbitrary units)', xlim=(0, t_b[-1]))
axa.set_title('(a) same $m$ and $k$, three dampings: the same frequency, different decay', loc='left', y=1.18)
axb.set_title('(b) same $Q$, three springs: different frequencies', loc='left', y=1.18)
for ax in (axa, axb):
    ax.legend(loc='lower left', bbox_to_anchor=(0.0, 1.0), ncol=3, frameon=False, fontsize=9)
plt.tight_layout()
plt.show()

**What to change.** On the line `# change these`, one parameter at a time:
1. Nothing first. In plot (a) look at the frequency of the three curves and at how long each one rings. In plot (b) look at the frequency and at the number of oscillations in $\tau_{0}$.
2. `b_values` in (a): halve each value. Which curves change and how?
3. `k_values` in (b): use 4, 16 and 64 N/m. What moves?
4. `m` (try 0.5 and 2 kg) with `b_values` as they are: what happens to the frequency and to the ringing?
5. `Q_b` (try 3 and 30): how many oscillations does each curve make in $\tau_{0}$?

**Question.** What sets the frequency of the ringing and what sets how long it lasts? How can we obtain a high $Q$?

<details>
<summary>Solution</summary>

In plot (a) the three curves oscillate at the same frequency, $f_{0} = 1.00$ Hz, although $b$ changes by a factor of four. They fade at different rates: $\tau_{0} = 1.59$, $3.17$ and $6.45$ s for $Q = 5$, $10$ and $20$. In plot (b) the frequency follows the spring: $0.50$, $1.00$ and $2.00$ Hz for $k = 9.87$, $39.5$ and $158$ N/m. Each curve of (b) makes $\tau_{0}f_{0} = Q/\pi = 3.2$ oscillations before it falls by $\mathrm{e}$, so a higher frequency means a shorter $\tau_{0}$: $6.4$, $3.2$ and $1.6$ s.

The **frequency** is set by the spring and the mass, $f_{0} = \sqrt{k/m}\,/2\pi$. The friction hardly moves it. The **decay** is set by the mass and the friction, $\tau_{0} = 2m/b$: half the friction gives twice the ringing. A heavier mass changes both: $m = 2$ kg gives $f_{0} = 0.71$ Hz and $\tau_{0}$ twice as long.

To obtain a high $Q = \sqrt{km}/b$ we reduce the friction $b$. At fixed $b$ we can also raise the stiffness or the mass. With $f_{0}$ fixed, $k = m\omega_{0}^{2}$ and $Q = m\omega_{0}/b$: a heavier mirror with the same friction has a higher $Q$. A detector uses both: fibers with very small friction and heavy mirrors.

</details>


## Part B · The transfer function from an impulse

We do not need to know $m$, $k$ and $b$ to know the system. We need only how it answers one kick.

### Task 2 · Kick the mirror and read $G$

**What we do.** A kick is a very short and strong push. The ground gives the mirror a unit impulse and the mirror starts from rest with the velocity $v_{0} = \omega_{0}^{2}$. It then rings freely: this motion is the impulse response $g(t)$. We find $g(t)$ by integrating Newton's second law from this start. Then we take its Fourier transform. The function `np.fft.rfft`, multiplied by the time step $\Delta t$, evaluates the sum that approximates the integral. This gives $G(f)$. We draw $|G|$ and its phase and put them on the formula

$$
G(\omega) = \frac{\omega_{0}^{2}}{\omega_{0}^{2} - \omega^{2} + \mathrm{i}\,\omega\,\omega_{0}/Q} .
$$

The formula comes from the equation of the forced mirror and is derived in the lecture. Here the kick finds it with no formula at all.

The steps of this task.

![A kick on the mirror, the system, the ringing g(t) in time, its transform G(f) in frequency.](figures/h02_task2_route.png)


In [ ]:
def transfer_function(f, m, k, b):
    """Return the transfer function G = X_out / X_in of the oscillator, from the formula.

    Parameters
    ----------
    f       : float or array, frequency (Hz)
    m, k, b : mass (kg), spring constant (N/m), damping coefficient (kg/s)

    Returns
    -------
    G : complex, same shape as f, no unit; |G| is the amplitude gain, angle(G) the phase (rad)
    """
    # G = w0^2 / (w0^2 - w^2 + i w w0 / Q), written with k = m w0^2 and b = m w0 / Q
    w = 2 * np.pi * f
    G = k / (k - m * w ** 2 + 1j * b * w)
    return G


def kick_response(t, m, k, b):
    """Return the impulse response g(t): the motion of the mirror after a unit kick of the ground.

    A unit impulse of the ground gives the mirror at rest the velocity v0 = w0^2 = k / m.
    Uses integrate_oscillator from task 1.

    Parameters
    ----------
    t       : array, times from the kick, evenly spaced (s)
    m, k, b : mass (kg), spring constant (N/m), damping coefficient (kg/s)

    Returns
    -------
    g : array of the same length as t, the impulse response (1/s)
    """
    # the kick: the mirror starts at x = 0 with the velocity k / m and rings freely
    g = integrate_oscillator(t, m, k, b, x0=0.0, v0=k / m)
    return g


def spectrum(g, dt):
    """Return the Fourier transform of a record g(t), as a function of the frequency.

    Parameters
    ----------
    g  : array, samples of g(t) every dt seconds
    dt : float, time step (s)

    Returns
    -------
    (f, G) : f frequencies (Hz) from 0 to 1/(2 dt); G the transform, in the unit of g times s
    """
    # rfft sums g[n] exp(-2 pi i f t[n]); times dt it approximates the integral of the transform
    G = np.fft.rfft(g) * dt
    f = np.fft.rfftfreq(len(g), dt)
    return f, G

In [ ]:
# change these: the mirror
m = 1.0       # mass of the mirror (kg)
k = 39.5      # spring constant (N/m): f0 = sqrt(k/m) / 2 pi, 1 Hz here
b = 0.63      # damping coefficient (kg/s): Q = sqrt(k m) / b, 10 here

# the record: 12 decay times, long enough for the ringing to fade; the time step sets the highest frequency
f0, Q, tau0 = oscillator_parameters(m, k, b)
dt = 0.005
t = np.arange(0, 12 * tau0, dt)

# route 1, in time: the kick and the ringing g(t)
g = kick_response(t, m, k, b)
# route 1, in frequency: the Fourier transform of g(t) is G(f); the formula is for comparison
f, G_kick = spectrum(g, dt)
G_formula = transfer_function(f, m, k, b)

# how far the transform of the kick is from the formula, between f0 / 10 and 10 f0
band = (f > f0 / 10) & (f < 10 * f0)
print(f"f0 = {f0:.3f} Hz   Q = {Q:.1f}   tau0 = {tau0:.2f} s   record of {t[-1]:.0f} s, {len(t)} samples")
print(f"largest relative difference in |G|: {np.max(np.abs(np.abs(G_kick[band]) / np.abs(G_formula[band]) - 1)):.1e}")
print(f"largest difference in the phase: {np.max(np.abs(np.angle(G_kick[band], deg=True) - np.angle(G_formula[band], deg=True))):.2f} degrees")

In [ ]:
# left: the ringing g(t) after the kick; right: |G| and the phase, the kick as dots on the formula
fig, axs = plt.subplot_mosaic([['g', 'amp'], ['g', 'ph']], figsize=(11, 5.5))
axs['g'].plot(t, g, color=P['SAGE'], lw=1.4, label='$g(t)$')
axs['g'].plot(t, (k / m) / (2 * np.pi * f0) * np.exp(-t / tau0), color=P['GREY_DARK'], lw=1.0, ls='--',
              label=r'$\pm\,(\omega_0^2/\omega_d)\,e^{-t/\tau_0}$')
axs['g'].plot(t, -(k / m) / (2 * np.pi * f0) * np.exp(-t / tau0), color=P['GREY_DARK'], lw=1.0, ls='--')
axs['g'].set(xlabel='time after the kick (s)', ylabel=r'$g(t)$ (1/s)', xlim=(0, 4 * tau0))
axs['g'].legend(loc='lower left', bbox_to_anchor=(0.0, 1.0), ncol=2, frameon=False)
axs['amp'].loglog(f[1:], np.abs(G_formula[1:]), color=P['AMBER'], lw=2.2, label='formula')
axs['amp'].loglog(f[1:], np.abs(G_kick[1:]), '.', color=P['INK'], ms=3, label='FFT of the kick')
axs['amp'].set(ylabel='$|G|$ (no unit)', xlim=(f0 / 10, 10 * f0), ylim=(5e-3, 3 * Q))
axs['amp'].legend(loc='lower left', bbox_to_anchor=(0.0, 1.0), ncol=2, frameon=False)
axs['ph'].semilogx(f[1:], np.angle(G_formula[1:], deg=True), color=P['AMBER'], lw=2.2)
axs['ph'].semilogx(f[1:], np.angle(G_kick[1:], deg=True), '.', color=P['INK'], ms=3)
axs['ph'].set(xlabel='frequency (Hz)', ylabel='phase of $G$ (degrees)', xlim=(f0 / 10, 10 * f0),
              yticks=[-180, -90, 0])
for name in ('amp', 'ph'):
    axs[name].axvline(f0, color=P['GREY_MID'], lw=0.8, ls=':')
plt.tight_layout()
plt.show()

**What to change.** On the line `# change these`, one parameter at a time:
1. Nothing first. Look at the ringing on the left and then at the dots on the right: do they lie on the formula?
2. `b` (try 0.21 and 2.1 kg/s): how does the ringing change and how does the peak of $|G|$ change?
3. `k` (try 10 and 160 N/m): where is the peak now?
4. `m` (try 0.5 and 2 kg): the same question.
Look at the printed differences each time.

**Question.** One kick is enough to know the system at every frequency. Why?

<details>
<summary>Solution</summary>

For $m = 1$ kg, $k = 39.5$ N/m and $b = 0.63$ kg/s the mirror has $f_{0} = 1.000$ Hz, $Q = 10.0$ and $\tau_{0} = 3.17$ s. The dots of the transform of the kick lie on the formula. Between $0.1f_{0}$ and $3f_{0}$ they differ by less than $10^{-3}$ in $|G|$ and by $0.01$ degrees in phase. The largest difference, $0.8\,\%$, is near $10f_{0}$ where $|G|$ is small and the 5 ms time step limits the transform.

With $b = 0.21$ kg/s ($Q = 30$) the peak is three times higher and the ringing three times longer. With $b = 2.1$ kg/s ($Q = 3$) the peak is hardly a peak. The spring and the mass move the peak: $k = 10$ and $160$ N/m give $f_{0} = 0.50$ and $2.01$ Hz and $m = 0.5$ and $2$ kg give $1.41$ and $0.71$ Hz.

**Why one kick is enough.** The Fourier transform of a very short kick is a constant: the kick contains all frequencies with the same weight. So it pushes the mirror at all frequencies at once. Because the system is linear the mirror answers each frequency separately, multiplied by $G$ at that frequency. The transform of the ringing is therefore $G(f)$ at every frequency. We found it without knowing $m$, $k$ and $b$. This is how a real system is measured: we kick it and record the ringing.

</details>


### Task 3 · Any input

**What we do.** Now the ground moves in a way we choose: two sines and a random motion. We compute the motion of the mirror in three ways. First, we integrate Newton's second law with this ground. Second, we convolve the input with $g(t)$ of task 2: each sample of the ground is a kick and the mirror adds the ringing of all the kicks. Third, we take the transform of the input, multiply it by $G(f)$ and go back to time:

$$
X_{\mathrm{out}}(f) = G(f)\,X_{\mathrm{in}}(f) .
$$

The plot shows the three outputs on the same axes. Below them we see the product in frequency: the amplitude spectrum of the input and the same spectrum multiplied by $|G|$.

The steps of this task.

![The motion of the ground, its transform, the product with G(f), the inverse transform: the motion of the mirror.](figures/h02_task3_route.png)


In [ ]:
def ground_motion(t, f1, f2, noise_rms, seed=3):
    """Return a ground motion: two sines plus a random motion below 4 Hz, switched on smoothly.

    Parameters
    ----------
    t         : array, times, evenly spaced (s)
    f1, f2    : frequencies of the two sines (Hz); their amplitudes are 1 and 0.5
    noise_rms : rms of the random motion (same unit as the sines)
    seed      : integer, seed of the random numbers

    Returns
    -------
    x_in : array of the same length as t, the ground position (arbitrary units)
    """
    # random motion: random numbers at the frequencies between 0.05 and 4 Hz, then back to time
    rng = np.random.default_rng(seed)
    freqs = np.fft.rfftfreq(len(t), t[1] - t[0])
    spec = np.zeros(len(freqs), dtype=complex)
    inside = (freqs > 0.05) & (freqs < 4)
    spec[inside] = rng.normal(size=inside.sum()) + 1j * rng.normal(size=inside.sum())
    noise = np.fft.irfft(spec, len(t))
    noise = noise_rms * noise / noise.std()
    # the two sines and the noise; the ground starts at rest, so the three methods share the start state
    x_in = np.sin(2 * np.pi * f1 * t) + 0.5 * np.sin(2 * np.pi * f2 * t) + noise
    x_in = x_in * np.sin(0.5 * np.pi * np.minimum(t / 5, 1)) ** 2
    return x_in


def output_by_convolution(x_in, g, dt):
    """Return x_out as the convolution of the input with the impulse response: a sum of kicks.

    Parameters
    ----------
    x_in : array, the ground position, samples every dt (arbitrary units)
    g    : array of the same length, the impulse response from kick_response (1/s)
    dt   : float, time step (s)

    Returns
    -------
    x_out : array of the same length, the position of the mirror (arbitrary units)
    """
    # each sample of x_in is a kick of size x_in * dt; the mirror adds the ringing of all the kicks
    x_out = np.convolve(x_in, g)[:len(x_in)] * dt
    return x_out


def output_by_product(x_in, dt, m, k, b):
    """Return x_out as the product G * X_in in frequency, then the inverse transform back to time.

    Uses transfer_function from task 2.

    Parameters
    ----------
    x_in    : array, the ground position, samples every dt (arbitrary units)
    dt      : float, time step (s)
    m, k, b : mass (kg), spring constant (N/m), damping coefficient (kg/s)

    Returns
    -------
    x_out : array of the same length as x_in, the position of the mirror (arbitrary units)
    """
    # the record is padded with zeros to twice its length, so the ringing at the end does not wrap around
    n = len(x_in)
    X_in = np.fft.rfft(x_in, 2 * n)
    f = np.fft.rfftfreq(2 * n, dt)
    # the product in frequency, then the inverse transform and the first n samples
    X_out = transfer_function(f, m, k, b) * X_in
    x_out = np.fft.irfft(X_out, 2 * n)[:n]
    return x_out

In [ ]:
# change these: the mirror and the ground
m = 1.0          # mass of the mirror (kg)
k = 39.5         # spring constant (N/m): f0 = sqrt(k/m) / 2 pi, 1 Hz here
b = 0.63         # damping coefficient (kg/s): Q = sqrt(k m) / b, 10 here
f1 = 0.7         # frequency of the first sine of the ground (Hz)
f2 = 2.3         # frequency of the second sine of the ground (Hz)
noise_rms = 0.5  # rms of the random ground motion, from 0.05 to 4 Hz (same unit as the sines)

# the ground motion for 80 s
dt = 0.01
t = np.arange(0, 80, dt)
x_in = ground_motion(t, f1, f2, noise_rms)
f0, Q, tau0 = oscillator_parameters(m, k, b)

# three ways to the same output: integrate Newton, convolve with g, multiply by G in frequency
x_newton = integrate_oscillator(t, m, k, b, x_in=x_in)
g = kick_response(t, m, k, b)
x_conv = output_by_convolution(x_in, g, dt)
x_prod = output_by_product(x_in, dt, m, k, b)

# the differences, as a fraction of the rms of the output
rms = x_newton.std()
print(f"f0 = {f0:.3f} Hz   Q = {Q:.1f}   rms of x_out = {rms:.3f}")
print(f"convolution against integration: largest difference {np.max(np.abs(x_conv - x_newton)) / rms:.1e} of the rms")
print(f"product against integration:     largest difference {np.max(np.abs(x_prod - x_newton)) / rms:.1e} of the rms")

In [ ]:
# top: the ground; middle: the three outputs on the same axes; bottom: the product in frequency
f_axis = np.fft.rfftfreq(len(t), dt)
X_in = np.fft.rfft(x_in) * dt
X_out = transfer_function(f_axis, m, k, b) * X_in
fig, axs = plt.subplots(3, 1, figsize=(10, 9))
axs[0].plot(t, x_in, color=P['BRICK'], lw=1.0)
axs[0].set(ylabel=r'$x_{\mathrm{in}}$ (arb. units)', xlabel='time (s)', xlim=(0, t[-1]))
axs[1].plot(t, x_newton, color=P['BLUE'], lw=3.5, alpha=0.6, label='integration of Newton')
axs[1].plot(t, x_conv, color=P['AMBER'], lw=1.8, ls='--', label='convolution with $g(t)$')
axs[1].plot(t, x_prod, color=P['INK'], lw=0.9, ls=':', label=r'product $G\,X_{\mathrm{in}}$')
axs[1].set(ylabel=r'$x_{\mathrm{out}}$ (arb. units)', xlabel='time (s)', xlim=(0, t[-1]))
axs[1].legend(loc='lower left', bbox_to_anchor=(0.0, 1.0), ncol=3, frameon=False)
axs[2].loglog(f_axis[1:], np.abs(X_in[1:]), color=P['BRICK'], lw=1.2, label=r'$|X_{\mathrm{in}}|$')
axs[2].loglog(f_axis[1:], np.abs(X_out[1:]), color=P['BLUE'], lw=1.2, label=r'$|G|\,|X_{\mathrm{in}}|$')
axs[2].axvline(f0, color=P['GREY_MID'], lw=0.8, ls=':')
axs[2].set(xlabel='frequency (Hz)', ylabel='amplitude spectrum (arb. units)', xlim=(0.05, 6))
axs[2].legend(loc='lower left', bbox_to_anchor=(0.0, 1.0), ncol=2, frameon=False)
plt.tight_layout()
plt.show()

**What to change.** On the line `# change these`, one parameter at a time:
1. Nothing first. The three outputs lie one on the other. Read the printed differences.
2. `f1` and `f2` (try 1.0 and 5.0 Hz): which sine does the mirror follow and which does it reject? Look at the lower plot.
3. `b` (try 0.063 and 2.1 kg/s): how does the ringing of the output change?
4. `k` (try 10 and 160 N/m): where does the mirror oscillate now?

**Question.** Why does the product in frequency give the same output as the integration? What do we gain from it?

<details>
<summary>Solution</summary>

The three outputs lie one on the other. The convolution differs from the integration by $1.0\times10^{-3}$ of the rms and the product by $0.8\times10^{-3}$. These are the errors of the time step and of the solver: the three methods describe the same linear system.

With $f_{1} = 0.7$ and $f_{2} = 2.3$ Hz the gain is $|G| = 1.94$ at $0.7$ Hz and $0.23$ at $2.3$ Hz. The sine near $f_{0}$ is amplified and the one above is reduced: the lower plot shows it. With $f_{2} = 5$ Hz the second sine is rejected, $|G| = 0.04$. A lower $b$ makes the ringing of the output longer and narrower in frequency. With $b = 0.063$ kg/s ($\tau_{0} = 32$ s) the product differs from the integration by $0.05$ of the rms. After the 80 s of zeros that we add the ringing is still $8\,\%$ of its size. It wraps around to the start of the record. A longer padding removes it. A stiffer or a softer spring moves the ringing to the $f_{0}$ of the new spring. For every other change the differences stay below $0.5\,\%$ of the rms.

**Why the product works.** The mirror is linear and does not change with time. A ground motion is a sum of kicks and each kick gives the ringing $g(t)$, so the output is the convolution of the input with $g$. The Fourier transform turns a convolution into a product: $X_{\mathrm{out}} = G\,X_{\mathrm{in}}$. Every frequency of the input is multiplied by one number.

**What we gain.** We need no integration of the equation for a new input: one transform, one product and one inverse transform. We can also go back: dividing the output by $G$ gives the input. And $G$ can be measured, as in task 2, even when we do not know the equation.

</details>


## Part C · The resonance and $Q$

The transfer function is a curve. Its peak is the resonance. We see what sets its height and its width and why we want a high $Q$.

### Task 4 · Why a high $Q$

**What we do.** We draw $|G(f)|$ and the phase for the three values of $Q$ of task 1 (a), with the same $f_{0}$. The ring on each peak marks its height, which is close to $Q$. The bar under it marks the width $\Delta f$ at half power, where $|G|$ is the peak divided by $\sqrt{2}$. For $Q \gg 1$ it is close to $f_{0}/Q$. The right plot shows the ringing of the same three oscillators. The table gives the product of the decay time and the width, $\tau_{0}\,\Delta\omega$ with $\Delta\omega = 2\pi\,\Delta f$. For every $Q$ it is close to 2:

$$
\tau_{0}\,\Delta\omega = \frac{2Q}{\omega_{0}}\,\frac{\omega_{0}}{Q} = 2 .
$$

> **The symbols**
>
> - **$\Delta f$** — the full width of the peak at half power, in hertz; $\Delta\omega = 2\pi\Delta f$

In [ ]:
def peak_numbers(f, gain):
    """Return the height, the frequency and the half-power width of the peak of a curve.

    Parameters
    ----------
    f    : array, frequencies in increasing order (Hz)
    gain : array, the amplitude gain |G| at those frequencies (no unit)

    Returns
    -------
    (height, f_peak, width) : height of the peak (no unit), its frequency (Hz) and the full
                              width where |G| is the height divided by sqrt(2) (Hz)
    """
    # the peak is the largest sample
    i_peak = np.argmax(gain)
    height = gain[i_peak]
    # the half-power level; the crossings on each side are found by linear interpolation
    level = height / np.sqrt(2)
    f_low = np.interp(level, gain[:i_peak + 1], f[:i_peak + 1])
    f_high = np.interp(level, gain[i_peak:][::-1], f[i_peak:][::-1])
    width = f_high - f_low
    numbers = (height, f[i_peak], width)
    return numbers

In [ ]:
# change these: the mirror, with the three dampings of task 1 (a)
m = 1.0                          # mass of the mirror (kg)
k = 39.5                         # spring constant (N/m): f0 = sqrt(k/m) / 2 pi, 1 Hz here
b_values = (1.26, 0.63, 0.31)    # damping coefficients (kg/s): Q = 5, 10, 20

# the gain |G| on a log grid around f0, and the ringing after a release from x = 1, for each damping
f0 = np.sqrt(k / m) / (2 * np.pi)
f = f0 * np.linspace(0.5, 1.5, 4000)
t = np.arange(0, 30, 0.01)
gains = [np.abs(transfer_function(f, m, k, b)) for b in b_values]
phases = [np.angle(transfer_function(f, m, k, b), deg=True) for b in b_values]
ringing = [integrate_oscillator(t, m, k, b, x0=1.0) for b in b_values]

# the numbers of each peak: height against Q, width against f0 / Q, and tau0 times the width in rad/s
print("   Q    height   f0/Q (Hz)   width (Hz)   tau0 (s)   tau0 * width (rad)")
for b, gain in zip(b_values, gains):
    f0_i, Q, tau0 = oscillator_parameters(m, k, b)
    height, f_peak, width = peak_numbers(f, gain)
    print(f"{Q:5.1f}  {height:7.2f}  {f0 / Q:10.4f}  {width:11.4f}  {tau0:9.2f}  {tau0 * 2 * np.pi * width:14.3f}")

In [ ]:
# left: |G| and the phase for the three Q; right: the ringing of the same three oscillators
colors = [P['BLUE'], P['BRICK'], P['SAGE']]
fig, axs = plt.subplot_mosaic([['amp', 'ring'], ['ph', 'ring']], figsize=(11, 6.5))
for b, gain, phase, x_i, c in zip(b_values, gains, phases, ringing, colors):
    f0_i, Q, tau0 = oscillator_parameters(m, k, b)
    height, f_peak, width = peak_numbers(f, gain)
    axs['amp'].semilogy(f, gain, color=c, lw=1.8, label=f'$Q$ = {Q:.0f}')
    # the height at f0 as a ring on the peak; the width as a bar at the half-power level
    axs['amp'].plot([f_peak], [height], 'o', mfc='none', mec=c, ms=8)
    axs['amp'].plot([f_peak - width / 2, f_peak + width / 2], [height / np.sqrt(2)] * 2, color=c, lw=2, marker='|', ms=10)
    axs['ph'].plot(f, phase, color=c, lw=1.8)
    axs['ring'].plot(t, x_i, color=c, lw=1.2)
    axs['ring'].plot(t, np.exp(-t / tau0), color=c, lw=1.0, ls='--')
    axs['ring'].plot(t, -np.exp(-t / tau0), color=c, lw=1.0, ls='--')
handles = axs['amp'].get_legend_handles_labels()[0]
handles += [Line2D([], [], marker='o', mfc='none', mec=P['INK'], ls='', label=r'height $\approx Q$'),
            Line2D([], [], color=P['INK'], lw=2, marker='|', ms=10, label=r'width $\approx f_0/Q$')]
axs['amp'].legend(handles=handles, loc='lower left', bbox_to_anchor=(0.0, 1.0), ncol=5, frameon=False, fontsize=9)
axs['amp'].set(ylabel='$|G|$ (no unit)', xlim=(0.5 * f0, 1.5 * f0), ylim=(0.5, 40))
axs['ph'].set(xlabel='frequency (Hz)', ylabel='phase of $G$ (degrees)', xlim=(0.5 * f0, 1.5 * f0), yticks=[-180, -90, 0])
axs['ring'].set(xlabel='time (s)', ylabel=r'$x_{\mathrm{out}}$ (arb. units)', xlim=(0, t[-1]))
axs['ring'].set_title('the same three oscillators, released from $x = 1$', loc='left', y=1.02, fontsize=10)
for name in ('amp', 'ph'):
    axs[name].axvline(f0, color=P['GREY_MID'], lw=0.8, ls=':')
plt.tight_layout()
plt.show()

**What to change.** On the line `# change these`, one parameter at a time:
1. Nothing first. For each curve compare the height with $Q$ and the width with $f_{0}/Q$ in the table. Then look at the ringing on the right.
2. `b_values` (try 3.15, 0.63 and 0.063 kg/s): look at $Q$ = 2, 10 and 100. Where does the height fall short of $Q$ and where does the width stop being $f_{0}/Q$?
3. `k` (try 20 and 80 N/m): what moves and what keeps its shape?

**Question.** What do we gain with a high $Q$ and what do we lose?

<details>
<summary>Solution</summary>

For $Q = 5.0$, $10.0$ and $20.3$ the height is $5.01$, $9.99$ and $20.28$. The width is $0.2026$, $0.1005$ and $0.0494$ Hz against $f_{0}/Q = 0.2005$, $0.1003$ and $0.0493$ Hz. The product $\tau_{0}\,\Delta\omega$ is $2.02$, $2.005$ and $2.001$. With $b = 3.15$ kg/s ($Q = 2$) the height is $2.06$ and the width is $0.54$ Hz against $f_{0}/Q = 0.50$ Hz. The product is $2.16$. For a low $Q$ the peak is wide and not symmetric and the rule is only rough. With $b = 0.063$ kg/s ($Q = 100$) the agreement is within $0.3\,\%$. For $k = 20$ and $80$ N/m the peak moves to $0.71$ and $1.42$ Hz and $Q$ changes to $7.1$ and $14.2$, so the height changes with it. The shape in $f/f_{0}$ depends on $Q$ alone.

**What we gain.** The response at $f_{0}$ is $Q$ times the response at low frequency. A weak force at $f_{0}$ produces a large motion.

**What we lose.** The peak is only $f_{0}/Q$ wide: the system answers only to a narrow band. And the ringing lasts $\tau_{0} = 2Q/\omega_{0}$, so $\tau_{0}\Delta\omega = 2$ always: a narrow peak and a long ringing are one fact, seen in frequency and in time. A system with a high $Q$ needs a long time to respond to a new signal and a long time to forget it.

</details>


## Part D · At the resonance and away from it

A setup is built with the $f_{0}$ and the $Q$ that its job needs. A bar wants the largest response in one narrow band, so it works at the resonance. A suspension wants the smallest response in a wide band, so it works far above it.

### Task 5 · Working at the resonance: a resonant bar

**What we do.** The first gravitational-wave detectors were resonant bars: metal cylinders of a few metric tons that ring when a wave passes (Allen and others 2000). We take round numbers: $m = 1000$ kg, $f_{0}$ near 900 Hz and $Q$ near $10^{6}$. A force $F$ at the frequency $f$ moves the bar by $|G|\,F/k$, the same shape as $|G|$ with the scale $1/k$. The cell draws the gain near $f_{0}$ for three dampings. It also draws the envelope of the ringing after a kick. It prints the response to one newton at $f_{0}$ and one hertz away.

**Uses** `transfer_function` from task 2, `oscillator_parameters` from task 1 and `peak_numbers` from task 4.

In [ ]:
# change these: the bar, with three dampings
m = 1000.0                       # mass of the bar (kg)
k = 3.2e10                       # spring constant (N/m): f0 = sqrt(k/m) / 2 pi, near 900 Hz
b_values = (57.0, 5.7, 0.57)     # damping coefficients (kg/s): Q near 1e5, 1e6, 1e7

# the gain |G| within 10 mHz of f0, and the envelope of the ringing after a kick, for each damping
f0 = np.sqrt(k / m) / (2 * np.pi)
f = f0 + np.linspace(-0.01, 0.01, 20001)
t = np.logspace(0, 4.5, 400)
far_f = f0 / 10
gains = [np.abs(transfer_function(f, m, k, b)) for b in b_values]

# the numbers of each bar; the response to a force F is |G| F / k, so per newton it is |G| / k
print("        Q   height   width (mHz)   f0/Q (mHz)   tau0 (s)   at f0 (m/N)   1 Hz away (m/N)   at f0/10 (m/N)")
for b, gain in zip(b_values, gains):
    f0_i, Q, tau0 = oscillator_parameters(m, k, b)
    height, f_peak, width = peak_numbers(f, gain)
    near = np.abs(transfer_function(f0 + 1.0, m, k, b)) / k
    far = np.abs(transfer_function(far_f, m, k, b)) / k
    print(f"{Q:9.2e}  {height:8.2e}  {1e3 * width:10.3f}  {1e3 * f0 / Q:11.3f}  {tau0:9.1f}  {height / k:11.2e}  {near:15.2e}  {far:15.2e}")

In [ ]:
# left: the gain near f0 for the three bars; right: the envelope of the ringing after a kick
colors = [P['BLUE'], P['BRICK'], P['SAGE']]
fig, (axa, axb) = plt.subplots(1, 2, figsize=(11, 4.5))
for b, gain, c in zip(b_values, gains, colors):
    f0_i, Q, tau0 = oscillator_parameters(m, k, b)
    axa.semilogy(1e3 * (f - f0), gain, color=c, lw=1.8, label=f'$Q$ = {Q:.1e}')
    axb.semilogx(t, np.exp(-t / tau0), color=c, lw=1.8)
    axb.axvline(tau0, color=c, lw=0.9, ls=':')
axa.axhline(1.0, color=P['GREY_MID'], lw=0.8, ls='--')
axa.set(xlabel='$f - f_0$ (mHz)', ylabel='$|G|$ (no unit)', ylim=(0.5, 3e7))
axa.legend(loc='lower left', bbox_to_anchor=(0.0, 1.0), ncol=3, frameon=False, fontsize=9)
axb.axhline(np.exp(-1), color=P['GREY_MID'], lw=0.8, ls=':')
axb.set(xlabel='time after the kick (s)', ylabel=r'amplitude $e^{-t/\tau_0}$', xlim=(1, 3e4), ylim=(0, 1.05))
plt.tight_layout()
plt.show()

**What to change.** On the line `# change these`:
1. Nothing first. Read in the table the response per newton at $f_{0}$ and one hertz away. On the left plot see how wide the peak is.
2. `b_values` as they are: compare the three bars. Which one has the largest response? Which one hears the widest band? Which one rings longest?
3. `m` and `k` together (try 500 kg and 1.6e10 N/m): $f_{0}$ stays the same. What changes in the table?

**Question.** The bar is very sensitive at its resonance. Where is it sensitive and how fast does it answer?

<details>
<summary>Solution</summary>

The bar has $f_{0} = 900.3$ Hz. For $b = 57$, $5.7$ and $0.57$ kg/s its quality factor is $9.9\times10^{4}$, $9.9\times10^{5}$ and $9.9\times10^{6}$. The width is $9.07$, $0.907$ and $0.091$ mHz, which is $f_{0}/Q$. The ringing time is $\tau_{0} = 35$, $351$ and $3509$ s. For one newton at $f_{0}$ the bar moves by $3.1\times10^{-6}$, $3.1\times10^{-5}$ and $3.1\times10^{-4}$ m. At $f_{0}/10$ it moves by $3.2\times10^{-11}$ m, which is $1/k$, whatever $b$ is: the response at the resonance is $Q$ times larger. One hertz away from $f_{0}$ the bar moves by $1.4\times10^{-8}$ m for every $b$. That is $2\,200$ times less than at $f_{0}$ for the middle bar. With $m = 500$ kg and $k = 1.6\times10^{10}$ N/m, $f_{0}$ is the same, $Q$ and $\tau_{0}$ are half and the width is twice.

**Where is the bar sensitive?** In a band $f_{0}/Q$ wide: less than a millihertz for $Q = 10^{6}$. One hertz away the gain is thousands of times smaller.

**How fast does it answer?** Slowly. The amplitude builds up and fades in $\tau_{0} = 2Q/\omega_{0}$, about 350 s. The gain $Q$, the band $f_{0}/Q$ and the time $\tau_{0}$ are related: ten times more friction gives a gain ten times smaller, a band ten times wider and a ringing ten times shorter. A bar hears a signal that lies in its band and lasts long enough to fill it.

</details>


### Task 6 · Working away from the resonance: the pendulum

**What we do.** A mirror of a detector is suspended as a pendulum. A pendulum of length $\ell$ is an oscillator with the spring constant $k = m g_{\oplus}/\ell$, so $f_{0} = \sqrt{g_{\oplus}/\ell}\,/2\pi$. For $\ell \approx 0.7$ m this is $0.6$ Hz. The detector measures from 10 to 100 Hz, far above $f_{0}$. There $|G| \simeq (f_{0}/f)^{2}$: the ground motion is isolated. Below $f_{0}$ the mirror follows the ground. The left plot shows one pendulum.

One stage is not enough. A chain of $N$ pendulums, one under the other, multiplies the transfer functions: $|G|^{N}$, which is $(f_{0}/f)^{2N}$ far above $f_{0}$. The right plot shows chains of 1 to 6 stages. The star is the gain we need at 10 Hz. It is the motion the mirror may have, $h L_{\mathrm{arm}} = 10^{-23} \times 4\ \mathrm{km} = 4\times10^{-20}\ \mathrm{m}/\sqrt{\mathrm{Hz}}$, divided by an assumed ground motion of $10^{-9}\ \mathrm{m}/\sqrt{\mathrm{Hz}}$ at 10 Hz, a model of a quiet site and not a measurement. The table says how many stages meet it and how long each one would have to be.

**Uses** `transfer_function` from task 2.

> **The symbols**
>
> - **$\ell$** — the length of one pendulum, in meters
> - **$g_{\oplus} = 9.81\ \mathrm{m/s^{2}}$** — the acceleration of gravity
> - **$N$** — the number of stages
> - **$h$** — the strain amplitude spectral density the interferometer must reach; **$L_{\mathrm{arm}}$** — the length of its arm

In [ ]:
G_EARTH = 9.81    # acceleration of gravity g (m/s^2)


def pendulum_parameters(m, ell, Q):
    """Return the spring constant, the damping and the natural frequency of a pendulum.

    A pendulum of length ell has the restoring force of a spring with k = m g / ell.

    Parameters
    ----------
    m   : float, mass hanging from it (kg)
    ell : float, length of the pendulum (m)
    Q   : float, quality factor (no unit)

    Returns
    -------
    (k, b, f0) : spring constant (N/m), damping coefficient (kg/s) and natural frequency (Hz)
    """
    # gravity plays the role of the spring; the damping follows from Q = sqrt(k m) / b
    k = m * G_EARTH / ell
    b = np.sqrt(k * m) / Q
    f0 = np.sqrt(G_EARTH / ell) / (2 * np.pi)
    return k, b, f0


def chain_gain(f, m, k, b, n_stages):
    """Return |G| of a chain of identical pendulums, one under the other.

    Uses transfer_function from task 2. The transfer functions of a chain multiply.

    Parameters
    ----------
    f        : float or array, frequency (Hz)
    m, k, b  : mass (kg), spring constant (N/m), damping coefficient (kg/s) of one stage
    n_stages : int, number of stages

    Returns
    -------
    gain : same shape as f, |G| to the power n_stages (no unit)
    """
    # the amplitude gain of one stage, to the power of the number of stages
    gain = np.abs(transfer_function(f, m, k, b)) ** n_stages
    return gain

In [ ]:
# change these: the pendulum and the requirement
m = 40.0                    # mass of the mirror (kg)
ell = 0.69                  # length of one pendulum (m)
Q = 1000.0                  # quality factor of one pendulum (no unit)
n_max = 6                   # largest number of stages drawn
ground_10 = 1.0e-9          # assumed ground motion at 10 Hz (m/sqrt(Hz)): a model of a quiet site, not a measurement
allowed_10 = 4.0e-20        # motion the mirror may have at 10 Hz (m/sqrt(Hz)): h = 1e-23 / sqrt(Hz) times 4 km

# the pendulum, its gain on a log grid, and the gain we need at 10 Hz
k, b, f0 = pendulum_parameters(m, ell, Q)
f = np.geomspace(0.05, 100, 2000)
gain_needed = allowed_10 / ground_10

# the numbers: one pendulum at three frequencies, then every chain against the requirement
print(f"f0 = {f0:.3f} Hz   gain needed at 10 Hz = {gain_needed:.1e}")
print(f"one pendulum: |G| at f0/100 = {chain_gain(f0 / 100, m, k, b, 1):.3f}: the mirror follows the ground")
for f_i in (10.0, 100.0):
    print(f"one pendulum: |G| at {f_i:5.0f} Hz = {chain_gain(f_i, m, k, b, 1):.2e}   (f0/f)^2 = {(f0 / f_i) ** 2:.2e}")
print("stages   gain at 10 Hz   meets it?   shortest length that meets it (m)")
for n in range(1, n_max + 1):
    gain_n = chain_gain(10.0, m, k, b, n)
    # the length for which (f0/10)^(2n) is the gain needed, from f0 = sqrt(g / ell) / 2 pi
    f0_needed = 10.0 * gain_needed ** (1 / (2 * n))
    ell_needed = G_EARTH / (2 * np.pi * f0_needed) ** 2
    print(f"{n:6d}   {gain_n:13.2e}   {'yes' if gain_n <= gain_needed else 'no ':>9}   {ell_needed:10.3g}")

In [ ]:
# left: one pendulum; right: chains of 1 to n_max pendulums against the gain needed at 10 Hz
fig, (axa, axb) = plt.subplots(1, 2, figsize=(11, 4.8))
axa.loglog(f, chain_gain(f, m, k, b, 1), color=P['BLUE'], lw=2.0, label='$|G|$ of one pendulum')
axa.loglog(f[f > f0], (f0 / f[f > f0]) ** 2, color=P['BRICK'], lw=1.4, ls='--', label=r'$(f_0/f)^2$')
axa.axvspan(10, 100, color=P['SAGE_LIGHT'], alpha=0.5, label='detector band')
axa.axvline(f0, color=P['GREY_MID'], lw=0.8, ls=':')
axa.text(0.06, 0.15, 'follows the ground', fontsize=9, color=P['GREY_DARK'])
axa.text(1.1, 3e2, r'isolates: $|G| \approx (f_0/f)^2$', fontsize=9, color=P['GREY_DARK'])
axa.set(xlabel='frequency of the ground (Hz)', ylabel='$|G|$ (no unit)', ylim=(1e-9, 3e3))
axa.legend(loc='lower left', bbox_to_anchor=(0.0, 1.0), ncol=3, frameon=False, fontsize=9)
palette = ['BLUE_MID', 'BLUE', 'TEAL', 'SAGE', 'AMBER', 'BRICK']
for n in range(1, n_max + 1):
    color = P[palette[(n - 1) % len(palette)]]
    axb.loglog(f, chain_gain(f, m, k, b, n), color=color, lw=1.6, label=f'N = {n}')
axb.plot([10], [gain_needed], '*', color=P['INK'], ms=12, ls='', label='gain needed at 10 Hz')
axb.axvline(f0, color=P['GREY_MID'], lw=0.8, ls=':')
axb.set(xlabel='frequency of the ground (Hz)', ylabel=r'$|G|^N$ (no unit)', xlim=(0.05, 100), ylim=(1e-30, 3e3))
axb.legend(loc='lower left', bbox_to_anchor=(0.0, 1.0), ncol=4, frameon=False, fontsize=8)
plt.tight_layout()
plt.show()

**What to change.** On the line `# change these`, in this order:
1. Nothing first. On the left plot find the three regimes: below $f_{0}$, at $f_{0}$ and above. Read $|G|$ at 10 and 100 Hz in the table.
2. On the right plot, which chain passes below the star? Read the table. `n_max` sets how many chains are drawn.
3. `m` (try 10 and 160 kg): does the isolation change?
4. `ell` (try 0.3 and 1.5 m): how do $f_{0}$ and the number of stages change?
5. `Q` (try 10 and 100000): what happens at $f_{0}$ and what happens at 10 Hz?

**Question.** Why do we work far above the resonance of a suspension? What do $m$, $\ell$ and $Q$ do to the isolation?

<details>
<summary>Solution</summary>

With $m = 40$ kg, $\ell = 0.69$ m and $Q = 1000$ the pendulum has $f_{0} = 0.600$ Hz. At $f_{0}/100$ the gain is $1.000$: the mirror follows the ground. At $f_{0}$ the gain is $Q$. At 10 Hz it is $3.6\times10^{-3}$ and at 100 Hz $3.6\times10^{-5}$, which is $(f_{0}/f)^{2}$. The gain needed at 10 Hz is $4.0\times10^{-11}$. The chains of 1 to 6 stages give $3.6\times10^{-3}$, $1.3\times10^{-5}$, $4.7\times10^{-8}$, $1.7\times10^{-10}$, $6.2\times10^{-13}$ and $2.2\times10^{-15}$. Four stages are a factor $4.3$ short and five are enough. Four stages of $0.69$ m would need a length of $0.99$ m, about $1$ m each, so that $f_{0} = 0.50$ Hz.

**Why far above the resonance.** Far above $f_{0}$ the inertia of the mass dominates over the spring and the mass stays still: $|G| \simeq (f_{0}/f)^{2} = k/(m\omega^{2})$. The detector band, 10 to 100 Hz, is in this regime. Below $f_{0}$ the spring dominates and the mirror follows the ground: a suspension isolates only above its resonance.

**What $m$, $\ell$ and $Q$ do.** The mass does nothing: $k = mg_{\oplus}/\ell$ grows with $m$ as the inertia does, so $f_{0}$ depends on $\ell$ alone. The values at 10 Hz are the same for $m = 10$, $40$ and $160$ kg. A longer pendulum has a lower $f_{0}$ and isolates more: $\ell = 0.3$ m gives $f_{0} = 0.91$ Hz and needs six stages, $\ell = 1.5$ m gives $0.41$ Hz and four stages are enough. The quality factor only sets the height of the peak at $f_{0}$: for $Q = 10$ or $10^{5}$ the gain at 10 Hz is the same to three digits. A high $Q$ gives a high peak at $0.6$ Hz, far below the detector band, where other means damp it. So the suspension has the resonance outside the band and works far above it.

</details>


## What to hand in

The answer to the question of each task, in two or three sentences, with the parameters you changed and what you saw:

1. **Task 1.** What sets the frequency, what sets the decay and how we obtain a high $Q$.
2. **Task 2.** Why one kick gives the transfer function at every frequency.
3. **Task 3.** Why the product $G\,X_{\mathrm{in}}$ gives the output for any input.
4. **Task 4.** What a high $Q$ gives and what we lose.
5. **Task 5.** Where and how fast a resonant bar is sensitive.
6. **Task 6.** Why a suspension works far above its resonance and how many stages it needs.

> **Where this goes next**
>
> One kick gave the transfer function of the oscillator and the transfer function gave the output for any input. A bar uses the resonance and a suspension keeps far from it. Both rest on linearity.

## Software

`numpy` throughout. `solve_ivp` integrates the equation of the oscillator and `np.fft` gives the transform of the kick and the product $G\,X_{\mathrm{in}}$. The oscillator, the loss and the pendulum isolation are treated as in (Saulson 2017). The transforms follow (Oppenheim et al. 1997). The resonant bars are described in (Allen and others 2000).

**References**

- Saulson (2017). *Fundamentals of Interferometric Gravitational Wave Detectors*.
- Oppenheim et al. (1997). *Signals and Systems*.
- Allen and others (2000). *First Search for Gravitational Wave Bursts with a Network of Detectors*.